# Daily Rank-Average Ensemble (LB 0.94650)

This notebook is updated most days of the competition. The method never changes: take genuinely independent public submissions/blends, convert each to a 0-1 percentile rank, and take a weighted average. What changes day to day is which sources go in and at what weight, based on two checks. First: does a new top-scoring public notebook actually add diversity? We measure the Spearman rank-correlation between its predictions and our current best blend. Above ~0.999 it is almost certainly remixing the same underlying models (adding it will not move the score); notably lower (we look for <=0.998) means it was built with a genuinely different feature/model pipeline and is worth blending in. Second: if it passes that check, sweep the mixing weight on the public leaderboard rather than guessing a weight. ROC-AUC only depends on rank order, so ~1.0-correlated sources cannot move the score no matter how they are weighted; genuinely diverse sources show a real, LB-measurable optimum.

## Current recipe (2026-09-14)

The current top of the Code tab (by Public Score) is dominated by one underlying file: Taeyang's lexsort output (https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master), which blends jazivxt's updated zoom-zoom anchor (90%) with a small stack and RealMLP (10%), applies deterministic boundary calibration, then breaks all ties with RealMLP via `np.lexsort`. We pulled this notebook's actual output directly and confirmed **0.94650 on the public LB** ourselves (not just the author's self-report). Every other notebook currently showing the same score (chinzorigtganbat, rohitt94, nathaliach) is a byte-for-byte or rank-identical repackaging of this same file (Spearman = 1.000000 against it) — no added diversity there.

The one source we found that is *not* rank-identical is nina2025's h-blend (5) (https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-5), which mixes in a small amount of her own separate blend history on top of the same lexsort file (Spearman 0.9987 against it — the most diverse thing currently available at the top of the leaderboard). We rank-averaged the two (70% Taeyang / 30% nina) and verified on the public LB: still **0.94650**, identical to Taeyang's file alone. So today's genuine finding is a validated negative: at this correlation level even the most different source currently visible adds no measurable score. We keep the two-source blend anyway (rather than just the raw single file) because it is a real, LB-checked combination and the weight is cheap insurance against future drift, not a guess.

`final = 0.70 * rank(Taeyang's lexsort master) + 0.30 * rank(nina2025's h-blend (5))`. LB-verified today: **0.94650**.

## Credits

None of the blended submissions were produced in this notebook. All modeling credit goes to their authors: taeyangg4 (S6E9 094649 Multi-Paradigm Lexsort Master, https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master — itself built on jazivxt's zoom-zoom pipeline and yekenot's RealMLP); nina2025 (PS-s6e9 h-blend (5), https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-5).

To reproduce: add s6e9-094649-multi-paradigm-lexsort-master and ps-s6e9-h-blend-5 as Notebook inputs (right sidebar, Add Input). Then run all cells.

In [ ]:
import pandas as pd
from scipy.stats import rankdata

## Load the source blends

In [ ]:
import glob
find_all = lambda pattern: sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
ID, TARGET = "id", "Will_Buy_EV"
taeyang = pd.read_csv(find_all("s6e9-094649-multi-paradigm-lexsort-master/submission.csv")[0]).set_index(ID)[TARGET]
nina = pd.read_csv(find_all("ps-s6e9-h-blend-5/submission.csv")[0]).set_index(ID)[TARGET].reindex(taeyang.index)
taeyang.head()

# Weighted rank average

In [ ]:
rank01 = lambda s: pd.Series(rankdata(s) / len(s), index=s.index)
final = 0.70 * rank01(taeyang) + 0.30 * rank01(nina)
final.head()

# Save submission

In [ ]:
submission = pd.DataFrame({ID: final.index, TARGET: final.to_numpy()})
submission.to_csv("submission.csv", index=False)
submission.head()